Алгоритмическая аугментация данных

Пайплайн:

1. Взять параллельные предложения на разных языках  
   - В данном случае датасет`facebook/flores`

2. Разбить их на слова (токены)  
   - Токенезируем через `spaCy`

3. Сопоставить слова между языками  
   - Используется мультиязычная библиотека `SimAlign`  

4. Алгоритмически расположить теги в en-предложении  
   - Скриптовые теги: `[Root.GetAdj]`, `[This.GetName]` — по POS (ADJ/NOUN/PROPN)  
   - Числа/переменные: `$NUM|2%$`, `$VAR$`, `$VALUE|Y$` — числа, имена, сущности (через POS)  
   - Иконки: `£icon£` — после существительных (NOUN-группы)  
   - Цветовые обёртки: `§Y ... §!` — вокруг важной фразы (NOUN/ADJ-окрестность)  

5. Построить структуру тегов вокруг токенов en  
   - Для каждого токена хранить:  
     - `before[i]`: теги до слова  
     - `replace[i]`: замена слова (если нужно)  
     - `after[i]`: теги после слова  
     - `color_spans`: список интервалов `(start, end, §Y, §!)`  
   - Потом один раз рендерить финальный список токенов, чтобы теги не затирали друг друга

6. Распространить расстановку тегов на другие языки  
   - Для каждого языка:  
     - через карту выравнивания найти целевые индексы для замены/вставки/обёрток  
     - построить такой же план тегов
     - отрендерить его в токены, затем склеить в строку

7. ???

8. PROFIT  

Правила распределения токенов можно доработать по факту
Проверим на языках с латиницей и китайском для проверки иероглифов

In [ ]:
# Датасет в виде скрипта, поэтому нужно использовать датасеты старой версии
!pip install datasets==3.6.0
!pip install simalign
!python -m spacy download en_core_web_sm
!python -m spacy download zh_core_web_sm

import random
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
import spacy
from datasets import load_dataset
from simalign import SentenceAligner




# Вероятности наличия тегов в предложении
TAG_PROBS = {
    "script": 0.4,  # скриптовые теги [Root.GetAdj] и т.п.
    "num":    0.5,  # $NUM|2%$
    "var":    0.3,  # $VAR$
    "value":  0.3,  # $VALUE|Y$
    "icon":   0.5,  # £sword_icon£
    "color":  0.7,  # §Y ... §!
}

SCRIPT_TAG_CANDIDATES = [
    "[Root.GetAdj]",
    "[This.GetName]",
    "[Root.Owner.GetSpecies.GetName]",
]

ICON_TAG    = "£sword_icon£"
NUM_TAG     = "$NUM|2%$"
VAR_TAG     = "$VAR$"
VALUE_TAG   = "$VALUE|Y$"
COLOR_OPEN  = "§Y"
COLOR_CLOSE = "§!"

# Языки FLORES
LANG_CODES = {
    "en": "eng_Latn",
    "ru": "rus_Cyrl",
    "zh": "zho_Hans",
    "pt": "por_Latn",
    "fr": "fra_Latn",
    "de": "deu_Latn",
}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 91.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('zh_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Ставим вероятности появления тегов и значения тегов

In [ ]:
TAG_PROBS = {
    "script": 0.4,  # вероятность вообще ставить хотя бы один скриптовый тег
    "num":    0.5,  # вероятность ставить $NUM|2%$
    "var":    0.3,  # вероятность ставить $VAR$
    "value":  0.3,  # вероятность ставить $VALUE|Y$
    "icon":   0.4,  # вероятность вставить £icon£
    "color":  0.7,  # вероятность добавить §Y ... §!
}

SCRIPT_TAG_CANDIDATES = [
    "[Root.GetAdj]",
    "[This.GetName]",
    "[Root.Owner.GetSpecies.GetName]"
]

ICON_TAG = "£sword_icon£"
NUM_TAG  = "$NUM|2%$"
VAR_TAG  = "$VAR$"
VALUE_TAG = "$VALUE|Y$"
COLOR_OPEN = "§Y"
COLOR_CLOSE = "§!"

Токенизация

In [ ]:
print("Loading spaCy models...")
nlp_en = spacy.load("en_core_web_sm")
nlp_zh = spacy.load("zh_core_web_sm")

print("Loading SimAlign (this may take a bit)...")
aligner = SentenceAligner(model="bert", token_type="bpe", matching_methods="mai")


nlp_by_lang = {
    "en": nlp_en,
    "ru": spacy.blank("ru"),
    "zh": nlp_zh,
    "pt": spacy.blank("pt"),
    "fr": spacy.blank("fr"),
    "de": spacy.blank("de"),
}

def tokenize_en(text: str):
    doc = nlp_en(text)
    tokens = [t.text for t in doc]
    return tokens, doc

def tokenize_by_lang(text: str, lang: str) -> List[str]:
    nlp = nlp_by_lang[lang]
    doc = nlp(text)
    return [t.text for t in doc]

Loading spaCy models...
Loading SimAlign (this may take a bit)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

2025-12-02 13:20:36,505 - simalign.simalign - INFO - Initialized the EmbeddingLoader with model: bert-base-multilingual-cased
INFO:simalign.simalign:Initialized the EmbeddingLoader with model: bert-base-multilingual-cased


Класс для определения структуры тегов

In [ ]:
@dataclass
class TagPlan:
    tokens: List[str]               # исходные токены
    replace: List[Optional[str]]    # чем заменить токены (или None, если оставить как есть)
    before: List[List[str]]         # теги перед токеном
    after: List[List[str]]          # теги после токена
    color_spans: List[Tuple[int,int,str,str]]  # (start, end, open_tag, close_tag)

def init_empty_plan(tokens: List[str]) -> TagPlan:
    n = len(tokens)
    return TagPlan(
        tokens=tokens,
        replace=[None] * n,
        before=[[] for _ in range(n)],
        after=[[] for _ in range(n)],
        color_spans=[],
    )


Построение плана тегов
1. **stage_scripts**  
   Выбирает один токен с ролью прилагательного/существительного/имени (ADJ/NOUN/PROPN)  
   и заменяет его случайным скриптовым тегом

2. **stage_numbers_and_vars**  
   - Находит числа и заменяет одним `$NUM|…$`.  
   - Находит имена собственные (PROPN) и заменяет одним `$VAR$`.  
   - Находит сущности/объекты и заменяет одним `$VALUE|Y$`.  
   Гарантирует, что теги не конфликтуют: каждый токен можно заменить только один раз.

3. **stage_icons**  
   Ищет существительные и вставляет `£icon£`.

4. **stage_color_wrap**  
   Выбирает фрагмент вокруг важного слова (NOUN/ADJ/PROPN) и оборачивает его в `§Y ... §!`.

Все стадии работают последовательно и только создают план (`replace/before/after/color_spans`),  
не изменяя текст напрямую — чтобы теги не перетирали друг друга.

In [ ]:
def available_indices(indices, taken):
    return [i for i in indices if i not in taken]

def is_number_like(tok: str) -> bool:
    s = tok.replace(",", "").replace(".", "")
    return s.isdigit() or any(ch.isdigit() for ch in tok)

def stage_scripts(plan: TagPlan, doc, probs: Dict):
    if random.random() > probs["script"]:
        return
    scriptable_idxs = [i for i, t in enumerate(doc) if t.pos_ in ("ADJ", "NOUN", "PROPN")]
    if not scriptable_idxs:
        return
    i = random.choice(scriptable_idxs)
    script_tag = random.choice(SCRIPT_TAG_CANDIDATES)
    plan.replace[i] = script_tag

def stage_numbers_and_vars(plan: TagPlan, doc, probs: Dict):
    taken_for_replace = {i for i, r in enumerate(plan.replace) if r is not None}

    # $NUM|2%$
    if random.random() <= probs["num"]:
        num_idxs = [i for i, tok in enumerate(plan.tokens) if is_number_like(tok)]
        num_idxs = available_indices(num_idxs, taken_for_replace)
        if num_idxs:
            i = random.choice(num_idxs)
            plan.replace[i] = NUM_TAG
            taken_for_replace.add(i)

    # $VAR$
    if random.random() <= probs["var"]:
        var_candidates = [i for i, t in enumerate(doc) if t.pos_ == "PROPN"]
        if not var_candidates:
            var_candidates = [i for i, t in enumerate(doc) if t.pos_ == "NOUN"]
        var_candidates = available_indices(var_candidates, taken_for_replace)
        if var_candidates:
            i = random.choice(var_candidates)
            plan.replace[i] = VAR_TAG
            taken_for_replace.add(i)

    # $VALUE|Y$
    if random.random() <= probs["value"]:
        value_candidates = [i for i, t in enumerate(doc) if t.pos_ in ("PROPN", "NOUN")]
        value_candidates = available_indices(value_candidates, taken_for_replace)
        if value_candidates:
            i = random.choice(value_candidates)
            plan.replace[i] = VALUE_TAG
            taken_for_replace.add(i)

def stage_icons(plan: TagPlan, doc, probs: Dict):
    if random.random() > probs["icon"]:
        return
    noun_idxs = [i for i, t in enumerate(doc) if t.pos_ == "NOUN"]
    if not noun_idxs:
        return
    i = random.choice(noun_idxs)
    plan.after[i].append(ICON_TAG)

def stage_color_wrap(plan: TagPlan, doc, probs: Dict):
    if random.random() > probs["color"]:
        return
    n = len(plan.tokens)
    if n == 0:
        return
    center_candidates = [i for i, t in enumerate(doc) if t.pos_ in ("NOUN", "ADJ", "PROPN")]
    if not center_candidates:
        center_candidates = list(range(n))
    center = random.choice(center_candidates)
    length = random.randint(1, 3)
    start = max(0, center - length // 2)
    end = min(n - 1, start + length - 1)
    plan.color_spans.append((start, end, COLOR_OPEN, COLOR_CLOSE))

Финальный рендер плана тегов

In [ ]:
def render_plan(plan: TagPlan) -> List[str]:
    n = len(plan.tokens)
    open_at = {i: [] for i in range(n + 1)}
    close_at = {i: [] for i in range(n + 1)}

    for (start, end, open_tag, close_tag) in plan.color_spans:
        open_at[start].append(open_tag)
        close_at[end + 1].append(close_tag)

    out = []
    for i in range(n):
        # сначала закрываем всё, что должно закрыться перед этим токеном
        for tag in close_at.get(i, []):
            out.append(tag)
        # открываем новые обёртки
        for tag in open_at.get(i, []):
            out.append(tag)
        # вставки до токена
        out.extend(plan.before[i])
        # сам токен или замена
        if plan.replace[i] is not None:
            out.append(plan.replace[i])
        else:
            out.append(plan.tokens[i])
        # вставки после токена
        out.extend(plan.after[i])

    # закрываем всё в конце
    for tag in close_at.get(n, []):
        out.append(tag)

    return out

**tag_english_with_plan**  
1) токенизируем исходное предложение
2) создаём пустой план  
3) по очереди добавляем теги
4) рендерим итоговые токены  
получаем: plan и размеченное EN-предложение

**build_alignment_map**  
Берём SimAlign → получаем карту выравнивания

**project_plan_to_target**  
1) создаём пустой план под целевой язык  
2) переносим теги en по выравниваниям  
3) рендерим план  
получаем размеченное предложение на другом языке

In [ ]:
def tag_english_with_plan(text: str) -> Tuple[TagPlan, List[str]]:
    tokens, doc = tokenize_en(text)
    plan = init_empty_plan(tokens)

    stage_scripts(plan, doc, TAG_PROBS)
    stage_numbers_and_vars(plan, doc, TAG_PROBS)
    stage_icons(plan, doc, TAG_PROBS)
    stage_color_wrap(plan, doc, TAG_PROBS)

    tagged_tokens = render_plan(plan)
    return plan, tagged_tokens

def build_alignment_map(src_tokens: List[str], tgt_tokens: List[str]) -> Dict[int, List[int]]:
    out = aligner.get_word_aligns(src_tokens, tgt_tokens)
    pairs = out["itermax"]
    align_map: Dict[int, List[int]] = {}
    for i, j in pairs:
        align_map.setdefault(i, []).append(j)
    return align_map

def project_plan_to_target(src_plan: TagPlan,
                           tgt_tokens: List[str],
                           align_map: Dict[int, List[int]]) -> TagPlan:
    n_tgt = len(tgt_tokens)
    tgt_plan = init_empty_plan(tgt_tokens)

    for i, tag in enumerate(src_plan.replace):
        if tag in SCRIPT_TAG_CANDIDATES:
            js = align_map.get(i, [])
            if not js:
                continue
            j = min(js)
            if tgt_plan.replace[j] is None:
                tgt_plan.replace[j] = tag
            else:
                tgt_plan.before[j].append(tag)

    for i, tag in enumerate(src_plan.replace):
        if tag not in (NUM_TAG, VAR_TAG, VALUE_TAG):
            continue
        js = align_map.get(i, [])
        if not js:
            continue
        j = min(js)
        if tgt_plan.replace[j] is None:
            tgt_plan.replace[j] = tag
        else:
            # если место занято скриптом — вставим перед
            tgt_plan.before[j].append(tag)

    for i, after_tags in enumerate(src_plan.after):
        if ICON_TAG in after_tags:
            js = align_map.get(i, [])
            if not js:
                continue
            j = max(js)
            tgt_plan.after[j].append(ICON_TAG)

    for (start, end, open_tag, close_tag) in src_plan.color_spans:
        js_all = []
        for k in range(start, end + 1):
            js_all.extend(align_map.get(k, []))
        if not js_all:
            continue
        js_all = sorted(set(js_all))
        j_start, j_end = js_all[0], js_all[-1]
        tgt_plan.color_spans.append((j_start, j_end, open_tag, close_tag))

    return tgt_plan

Загрузка датасета

In [ ]:
def load_flores_subset(num_sentences=10):
    ds = load_dataset("facebook/flores", "all")
    split = ds["devtest"]
    results = []
    for i in range(num_sentences):
        row = split[i]
        item = {}
        for lang, code in LANG_CODES.items():
            item[lang] = row[f"sentence_{code}"]
        results.append(item)
    return results


Собираем в финальную функцию

In [ ]:
def augment_multilingual_example(example: Dict[str, str]) -> Dict[str, str]:
    base_lang = "en"
    base_text = example[base_lang]

    # Тегируем английский
    src_plan, tagged_en_tokens = tag_english_with_plan(base_text)
    result = {base_lang: " ".join(tagged_en_tokens)}

    # Для каждого другого языка
    for lang, text in example.items():
        if lang == base_lang:
            continue
        tgt_tokens = tokenize_by_lang(text, lang)
        align_map = build_alignment_map(src_plan.tokens, tgt_tokens)
        tgt_plan = project_plan_to_target(src_plan, tgt_tokens, align_map)
        tagged_tgt_tokens = render_plan(tgt_plan)
        result[lang] = " ".join(tagged_tgt_tokens)

    return result

In [ ]:
def main_test():
    random.seed(42)
    examples = load_flores_subset(num_sentences=10)

    for idx, ex in enumerate(examples):
        print("=" * 80)
        print(f"Example {idx}")
        print("- ORIGINAL:")
        for lang in LANG_CODES.keys():
            print(f"[{lang}] {ex[lang]}")

        aug = augment_multilingual_example(ex)

        print("- AUGMENTED:")
        for lang in LANG_CODES.keys():
            print(f"[{lang}] {aug[lang]}")

if __name__ == "__main__":
    main_test()

README.md: 0.00B [00:00, ?B/s]

flores.py: 0.00B [00:00, ?B/s]

The repository for facebook/flores contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/facebook/flores.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

Example 0
- ORIGINAL:
[en] "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
[ru] "Теперь у нас есть четырёхмесячные мыши, у которых больше нет диабета", — добавил он.
[zh] 他补充道：“我们现在有 4 个月大没有糖尿病的老鼠，但它们曾经得过该病。”
[pt] "Agora temos ratos de 4 meses de idade que não são diabéticos e que antes eram diabéticos,"complementou.
[fr] « Nous avons à présent des souris de 4 mois qui ne sont pas diabétiques alors qu'elles l'étaient auparavant », a-t-il ajouté.
[de] „Wir haben jetzt 4 Monate alte Mäuse, die Diabetes hatten und jetzt keinen mehr haben“, fügte er hinzu.
- AUGMENTED:
[en] " We now have $NUM|2%$ - $VAR$ - old mice that are non - diabetic that used to be diabetic , " he added .
[ru] " Теперь у нас есть $VAR$ $NUM|2%$ мыши , у которых больше нет диабета " , — добавил он .
[zh] 他 补 充道 ： “ 我们 现在 有 $NUM|2%$ 个 $VAR$ 大 没有 糖尿病 的 老鼠 ， 但 它们 曾经 得 过 该病 。 ”
[pt] " Agora temos ratos de $NUM|2%$ $VAR$ de idade que não são diabéticos e que antes eram diabéticos,"c

В принципе результаты аугментации неплохие. Теги хорошо переносятся между языками, и в целом консистентность сохраняется. После доработки правил появления токенов всё будет отлично (учитывая что сейчас повяление токенов для чисел и [] может ломать грамматику, нужно будет подумать на д определением того, какой член предложения надо заменять на тег).
В целом можно попросить LLM дополнительно пройтись и поправить согласованность тегов.

По крайней мере это дешевле по запросам чем чистая генерация через LLM.